# Train YOLOv5 on full BDD100K (GPU, memory-safe)

Re-run of the full BDD100K training, fixed after the previous attempt (`bdd100k_vehicle_100e_bs8`) trained to poor results (mAP@0.5 ≈ 0.11 after 100 epochs).

**What this notebook fixes / guards against:**
1. The training script now resolves `bdd100k_vehicle.yaml` robustly and **fails loudly** if it can't find the file, instead of YOLOv5 silently doing something unexpected with a bad path.
2. A hard check for CUDA before training starts — the previous run had a session where CUDA was not actually available, which either crashed (`--device 0` with no GPU) or silently fell back to CPU.
3. Memory-safety rules requested for this run (~70k train + 10k val images, must stay under 32 GB system RAM):
   - `--batch-size` capped at **8 or 16** (configurable below, default 8)
   - `--workers 2` always — fewer dataloader threads pulling images into RAM at once
   - **No `--cache` / `--cache ram` — ever.** That flag pre-loads the *entire* image set into RAM (or a decoded array), which for 70k+ BDD100K images will blow past 32 GB. This notebook never passes it.

Run outputs are written directly into the reorganized `runs/train/bdd100k/`, `runs/val/bdd100k/`, `runs/detect/bdd/` folders so no manual re-sorting is needed afterward.

In [ ]:
!nvidia-smi

## 1. Set working directory

In [ ]:
import os, sys

project = os.path.expanduser("~/Vehicle-Detection-and-Distance-Estimation/yolov5")
os.chdir(project)

print("Working directory:", os.getcwd())
print("Python:", sys.executable)

## 2. Locate the dataset config

The previous run used `../dataset/dataset/bdd100k_vehicle.yaml` (double `dataset/`), which does not exist
in this repo's local layout — only `../dataset/bdd100k_vehicle.yaml` does. This cell checks both known
layouts and **raises an error immediately** if neither is found, instead of letting `train.py` fail deep
inside its own dataset-loading code with a less obvious error.

In [ ]:
from pathlib import Path

candidate_paths = [
    Path("../dataset/bdd100k_vehicle.yaml"),
    Path("../dataset/dataset/bdd100k_vehicle.yaml"),
]

DATA_YAML = next((p for p in candidate_paths if p.exists()), None)

if DATA_YAML is None:
    checked = "\n".join(f"  - {p.resolve()}" for p in candidate_paths)
    raise FileNotFoundError(
        "bdd100k_vehicle.yaml not found. Checked:\n" + checked
    )

print("Using data config:", DATA_YAML.resolve())

## 3. Verify GPU is actually available

Do **not** proceed to training if this cell fails — that was the root cause of the previous run silently training on CPU / crashing on `--device 0`.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available in this session. Restart with a GPU-enabled "
        "runtime/kernel before training — do not proceed on CPU."
    )

props = torch.cuda.get_device_properties(0)
print("PyTorch  :", torch.__version__)
print("CUDA     :", torch.version.cuda)
print("GPU      :", props.name)
print(f"VRAM     : {props.total_memory / 1024**3:.1f} GB")

## 4. Check system RAM headroom

Baseline check before training — with `--cache` never used, RAM usage should stay roughly flat (just the `--workers 2` prefetch buffers), not climb toward the full dataset size.

In [ ]:
import psutil

mem = psutil.virtual_memory()
print(f"System RAM total     : {mem.total / 1024**3:.1f} GB")
print(f"System RAM available : {mem.available / 1024**3:.1f} GB")

if mem.total / 1024**3 > 32:
    print("\nNote: more than 32GB physical RAM detected, but this notebook still "
          "intentionally avoids --cache to stay well within the 32GB budget.")

## 5. Training config

`BATCH_SIZE`: start at **8**. Only raise to **16** if step 4 shows solid RAM headroom AND VRAM in step 3 is
comfortably above what `batch=8` used in the previous run — bump it back down to 8 if you see CUDA OOM or
system RAM climbing during training.

`WORKERS` is fixed at 2 per the requirement — do not raise this for this dataset size.

In [ ]:
IMG_SIZE = 640
BATCH_SIZE = 8       # 8 or 16 only — see note above
EPOCHS = 100
WORKERS = 2           # fixed, do not increase
PROJECT_DIR = "runs/train/bdd100k"
RUN_NAME = "bdd100k_vehicle_100e_v2"

DATA_YAML_STR = str(DATA_YAML)

print(f"batch={BATCH_SIZE}  workers={WORKERS}  epochs={EPOCHS}")
print(f"data={DATA_YAML_STR}")
print(f"output -> {PROJECT_DIR}/{RUN_NAME}")

## 6. Train

No `--cache` flag anywhere below — intentional, per the memory requirement.

In [ ]:
!{sys.executable} -W ignore train.py     --img {IMG_SIZE}     --batch {BATCH_SIZE}     --epochs {EPOCHS}     --data {DATA_YAML_STR}     --weights yolov5s.pt     --workers {WORKERS}     --device 0     --project {PROJECT_DIR}     --name {RUN_NAME}

### If training gets interrupted

Resume from the last checkpoint instead of restarting from epoch 0 (uncomment and run):

In [ ]:
# !{sys.executable} train.py --resume {PROJECT_DIR}/{RUN_NAME}/weights/last.pt

## 7. Validate best checkpoint

In [ ]:
!{sys.executable} -W ignore val.py     --weights {PROJECT_DIR}/{RUN_NAME}/weights/best.pt     --data {DATA_YAML_STR}     --img {IMG_SIZE}     --batch {BATCH_SIZE}     --device 0     --project runs/val/bdd100k     --name {RUN_NAME}

## 8. Sample detections (optional)

Points at the full BDD100K val folder (10,000 images) by default — point `--source` at a smaller folder/single image if you just want a quick visual check.

In [ ]:
!{sys.executable} -W ignore detect.py     --weights {PROJECT_DIR}/{RUN_NAME}/weights/best.pt     --source ../dataset/raw/BDD100K/bdd100k/bdd100k/images/100k/val     --img {IMG_SIZE}     --device 0     --project runs/detect/bdd     --name {RUN_NAME}